# Spatial-cell analysis

Use the remote `~/.virtualenvs/rfmapping` kernel on `hhw9l84`. This notebook saves four `.rfmap` files: the full angle × distance matrices and 360° bearing curves for distance ≤8 cm, 8 < distance ≤16 cm, and distance >16 cm. Each curve sums the existing matrix bins whose distance centers fall in that band. Open any output in `spatial_cell_plotting.ipynb`.


In [6]:
from importlib import reload
from pathlib import Path

import spatial_cell_analysis as analysis
analysis = reload(analysis)


## Recording and camera output

Set exactly one output bool to `True`; the camera type is never inferred. Saved camera timestamps are used when available. Otherwise ADC pulse midpoints are read as in the tuning-curve notebooks. The ADC channel is zero-based, and the threshold is in raw int16 ADC units.


In [7]:
analysis.recording_root = Path("/mnt/senzailab/Kai/#Recording/m20")
analysis.date = "260918"
analysis.recording_number = 11
analysis.probe_name = "A"
analysis.phase_key = "baseline"

basler_output = True
optihub2_output = False
camera_input_channel = 1
camera_ttl_threshold = 14000

unit_ids = None  # None: all good units; or a list such as [7, 9].
workers = 4
result_path = (
    analysis.recording_root / analysis.date
    / f"{analysis.date}_{analysis.recording_number}"
    / "data" / "spatial_cells" / f"Probe{analysis.probe_name}" / analysis.phase_key
    / "egocentric_rate_map.rfmap"
)


## Arena and map parameters

The pose CSV must contain `frame`, `center_x`, `center_y`, and `hd_deg` in the existing image-coordinate convention. Raw Motive CSV exports must first be processed into these columns.

For this setup, Basler uses the midpoint of each complete **low** pulse (opto-coupled `ExposureActive`, without additional `LineInverter` inversion); OptiHub2 uses **high** pulse midpoints. Low intervals at the ADC boundaries are not Basler frames. These are explicit presets, not signal detection. See [Basler output levels](https://docs.baslerweb.com/line-status#opto-coupled-output-line) and [LineInverter](https://docs.baslerweb.com/line-inverter).

Only OptiHub2 permits the single trailing Motive frame without a TTL, as in `tuning_curves.ipynb`.


In [8]:
analysis.x_min, analysis.x_max = 370, 920
analysis.y_min, analysis.y_max = 210, 760
analysis.rig_size_cm = 41
analysis.cm_per_px = analysis.rig_size_cm / (analysis.x_max - analysis.x_min)

analysis.theta_bin_deg = 6
analysis.number_of_distance_bins = 20
analysis.egocentric_smoothing_sigma = 5


## Analyze and save

Save four `.rfmap` files after all selected units finish. `result_path` names the full matrix; the three bearing files append `_0-8`, `_8-16`, and `_16-` to its stem. Rerunning replaces all four files; change `result_path` to keep another version. `result_paths` lists all four saved paths.


In [9]:
result_paths = analysis.run_analysis(
    output=result_path,
    units=unit_ids,
    workers=workers,
    basler_output=basler_output,
    optihub2_output=optihub2_output,
    camera_input_channel=camera_input_channel,
    camera_ttl_threshold=camera_ttl_threshold,
)


basler: 72222 camera frames, 25.000 Hz (adc_exposure_midpoint); 72222 valid pose frames in baseline
Saved 481 tuning matrices: /mnt/senzailab/Kai/#Recording/m20/260918/260918_11/data/spatial_cells/ProbeA/baseline/egocentric_rate_map.rfmap
Saved 481 tuning matrices: /mnt/senzailab/Kai/#Recording/m20/260918/260918_11/data/spatial_cells/ProbeA/baseline/egocentric_rate_map_0-8.rfmap
Saved 481 tuning matrices: /mnt/senzailab/Kai/#Recording/m20/260918/260918_11/data/spatial_cells/ProbeA/baseline/egocentric_rate_map_8-16.rfmap
Saved 481 tuning matrices: /mnt/senzailab/Kai/#Recording/m20/260918/260918_11/data/spatial_cells/ProbeA/baseline/egocentric_rate_map_16-.rfmap
